# Trassenfinder API Query

Collects the training dataset for the night train energy model from Deutsche
Bahn's Trassenfinder API: energy consumption, route distance and technical
travel time for every route segment, queried once per standard train
composition.

Runs top to bottom — no manual steps, no cells to skip.

## Two route sources

| Source | Routes | What it is |
|---|---|---|
| `ontd` | 95 | Real German night-train segments from the ONTD workbook — station to station, as actually operated |
| `synthetic` | 100 | Generated point-to-point pairs, sampled for geographic and length diversity |

They cover different parts of the design space. ONTD segments are mostly short
(median 78 km air distance) because real night trains stop frequently; the
generated set is mostly long (median 355 km) and spreads across N-S, E-W and
diagonal axes. Running both lets us check whether coefficients fitted on
operational segments hold on arbitrary station pairs — if they diverge, the
model is picking up something about how night trains are routed rather than
about the physics of moving a train.

Each source is collected and saved separately, then combined with a `source`
column so the regression notebook can fit on either or both.

## Pipeline

1. Load compositions and both route sources, normalise to a common schema
2. Build the request payload template
3. Define the query function
4. Resolve every station once (pre-flight)
5. Collect each source
6. Save per source, plus a combined file
7. Compare the two samples and quality check

## API

Deutsche Bahn Trassenfinder, `POST /api/web/routen/suche`. No authentication.
Stations are identified by DS100 code.

**Runtime:** roughly 1,560 collection requests plus around 150 for the
pre-flight. Budget 30–50 minutes.

## 1. Setup

Both sources are normalised to the same five columns — `route_name`,
`start_stop_name`, `start_ds100`, `end_stop_name`, `end_ds100` — so everything
downstream is source-agnostic.

Two corrections are applied before querying:

- **`DS100_CORRECTIONS`** — codes Trassenfinder rejects in both mother and child
  form. `KKSU` for Köln Süd is not a valid Betriebsstelle; the correct code is
  `KKS`. Add here rather than editing the source CSVs, so the correction stays
  visible and upstream fixes drop in without conflict.
- **Segments without a DS100** — Lörrach Autoreisezug Terminal has no code in
  the ONTD export and cannot be queried.

In [ ]:
import copy

import pandas as pd
import requests

from data_sources import DATA_DIR, SEED_DIR, SOURCES_DIR, source_input

SOURCES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
SEED_DIR.mkdir(parents=True, exist_ok=True)

# DS100 codes that Trassenfinder rejects in both mother and child form.
DS100_CORRECTIONS = {
    "KKSU": "KKS",  # Köln Süd
}

SCHEMA = ["route_name", "start_stop_name", "start_ds100", "end_stop_name", "end_ds100"]


def load_ontd():
    """Real night-train segments — already in the target schema."""
    df = pd.read_csv(source_input("routes_ontd.csv"))
    return df[SCHEMA]


def load_synthetic():
    """Generated station pairs — renamed into the target schema.

    The file has no route identifier, so one is minted per row; segment_id
    downstream needs it to be unique.
    """
    df = pd.read_csv(source_input("routes_synthetic.csv"))

    df = df.rename(
        columns={
            "start_DS100": "start_ds100",
            "end_DS100": "end_ds100",
            "start_name": "start_stop_name",
            "end_name": "end_stop_name",
        }
    )
    df["route_name"] = [f"SYN-{i:03d}" for i in range(1, len(df) + 1)]

    return df[SCHEMA]


SOURCES = {
    "ontd": load_ontd,
    "synthetic": load_synthetic,
}

compositions = pd.read_csv(source_input("compositions.csv"))

route_sets = {}
for name, loader in SOURCES.items():
    df = loader()
    df = df.replace({"start_ds100": DS100_CORRECTIONS, "end_ds100": DS100_CORRECTIONS})

    dropped = df[df[["start_ds100", "end_ds100"]].isna().any(axis=1)]
    df = df.dropna(subset=["start_ds100", "end_ds100"]).reset_index(drop=True)
    route_sets[name] = df

    print(f"{name:10s} {len(df):4d} routes  ({len(dropped)} dropped, no DS100)")
    for _, row in dropped.iterrows():
        print(f"           - {row['start_stop_name']} -> {row['end_stop_name']}")

print("\nCompositions:", len(compositions))
print("Total requests:", sum(len(df) for df in route_sets.values()) * len(compositions))

# Station pairs present in both sources would be queried twice with identical
# settings, so the duplicate is worth knowing about before the run.
pairs = {
    name: set(zip(df["start_ds100"], df["end_ds100"]))
    for name, df in route_sets.items()
}
overlap = pairs["ontd"] & pairs["synthetic"]
print("Overlapping station pairs:", len(overlap), sorted(overlap) if overlap else "")

url = "https://trassenfinder.de/api/web/routen/suche"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Origin": "https://trassenfinder.de",
    "Referer": "https://trassenfinder.de/",
}

## 2. Request payload template

Passenger long-distance traffic with a locomotive (`spfv_lok`). Route and
composition fields are overwritten per request; everything else is held
constant across both sources so that energy differences are attributable to
route and train, not to search settings.

Three settings shape the result and matter when interpreting the data:

- `schnellfahrstrecken_meiden` — high-speed lines avoided, as night trains
  generally do not use them (night-time maintenance windows)
- `gewichtung_parameter` — the route returned is a 40/30/30 compromise between
  distance, time and energy, not the shortest path
- `bremshundertstel`, `streckenklasse`, `bremsstellung` — fixed for all
  compositions, so they act as a level effect rather than a difference between
  trains

In [ ]:
PAYLOAD_TEMPLATE = {
    "infrastruktur_id": 8,
    "sucheinstellungen": {
        "verkehrsart": "spfv_lok",
        "an_abzeit": "2026-08-12T20:00:00+02:00",
        "zeitvorgabe_typ": "abzeit",
        "optimierungsvarianten_berechnen": True,
        "richtungswechsel_zulaessig": True,
        "rangierfahrt_zulaessig": False,
        "vermeidung_parameter": {
            "ueberlastete_meiden": False,
            "sbahnen_meiden": True,
            "nebenbahnen_meiden": False,
            "schnellfahrstrecken_meiden": True,
            "knotenbahnhoefe_meiden": True,
            "notbremsueberbrueckung_meiden": False,
            "wirbelstrombremse_meiden": False,
            "eingleisige_strecken_meiden": False,
            "strecken_mit_vorrang_sgv_meiden": False,
            "strecken_mit_vorrang_spv_meiden": False,
        },
        "initiale_sperrungen_beruecksichtigen": True,
        "wendezeit_min": 30,
        "mit_realistischen_fahrzeiten_optimieren": True,
        "verkehrshalte_nur_an_bahnsteigen": False,
        "einschraenkungen_beachten": True,
        "manueller_fahrzeitzuschlag_prozent": 0,
        "einzelgrenzlastberechnung_zulaessig": False,
        "bauzuschlaege_beachten": False,
        "laengenabhaengige_grenzlasten_verwenden": False,
        "tpn_triebfahrzeugbezeichnung_anzeigen": False,
        "gewichtung_parameter": {
            "streckenlaenge_prozent": 40,
            "fahrzeit_prozent": 30,
            "energie_prozent": 30,
        },
        "zusatzkosten_parameter": {
            "energiebezugspreis_euro_pro_kwh": 0.18,
            "rueckspeisung_euro_pro_kwh": 0.09,
            "kosten_besetzte_tfz_inkl_personal_euro_pro_h": 150,
            "kosten_unbesetzte_tfz_euro_pro_h": 80,
            "kosten_wagenzug_euro_pro_h": 150,
            "kostenpauschale_ungekuppelt_nachschieben_euro": 999,
            "zusaetzlicher_energieverbrauch_pro_wagen_kw": 0,
            "energieverbrauch_hilfsbetriebe_und_wagen_beachten": True,
        },
    },
    "wegpunkte": [
        {
            "zugcharakteristik": {
                "bremshundertstel": 70,
                "bremsstellung": "P",
                "aktive_neigetechnik": False,
                "kupplungsbauart": "kn450",
                "dla_u_profile": [],
                "fuehrendes_fahrzeug": "lokomotive",
                "kv_profil": {"p": "N", "c": "N"},
                "nachschiebeart": "ohne",
                "streckenklasse": "D4",
                "traktionsartwechsel": False,
                "triebfahrzeug": {
                    "hauptnummer": "6185",
                    "unternummer": 2,
                    "kennung": "L",
                    "kennung_wert": 80,
                },
                "vorspannart": "ohne",
                "wagenzuglaenge_m": 600,
                "wagenzugmasse_t": 1200,
                "wagenanzahl": 0,
                "v_max": 100,
                "zugbeeinflussung_parameter": {
                    "etcs_system_version": "ohne",
                    "lzb": True,
                    "pzb": True,
                },
            },
            "betriebsstelle": {"ds100": "AH", "mutter": True},
        },
        {"betriebsstelle": {"ds100": "MH", "mutter": True}},
    ],
    "nutzer_sperrungen": [],
}

print("✓ Payload template ready")

## 3. Query function

`mutter` is a fixed property of each Betriebsstelle, not a free choice:
Trassenfinder rejects a Mutterbetriebsstelle sent as a child and a child sent
as a mother, both with HTTP 400. Köln Hbf (`KK`) is only valid as a mother;
Aachen Hbf (`KA`), Koblenz (`KKO`), Freiburg (`RF`), Hanau (`FH  N`), Erfurt
(`UE  P`) and Berlin Hbf (`BLS`) only as children.

The flag is therefore resolved per station and cached across both sources.

In [ ]:
# Resolved "mutter" flag per DS100, filled by resolve_station() and reused by
# every subsequent request for that station, in either source.
MUTTER_CACHE: dict[str, bool] = {}


class TrassenfinderError(RuntimeError):
    """Carries the API response body, which raise_for_status() would discard."""


def build_payload(start_ds100, end_ds100, composition, start_mutter, end_mutter):
    """Return a request body for one route segment and one composition."""
    body = copy.deepcopy(PAYLOAD_TEMPLATE)

    body["wegpunkte"][0]["betriebsstelle"] = {
        "ds100": str(start_ds100),
        "mutter": start_mutter,
    }
    body["wegpunkte"][1]["betriebsstelle"] = {
        "ds100": str(end_ds100),
        "mutter": end_mutter,
    }

    zug = body["wegpunkte"][0]["zugcharakteristik"]
    zug["triebfahrzeug"]["hauptnummer"] = str(
        composition["trassenfinder_triebfahrzeug_hauptnummer"]
    )
    zug["wagenzuglaenge_m"] = float(composition["coaches_length_m_wagenzuglaenge"])
    zug["wagenzugmasse_t"] = float(
        composition["coaches_gross_weight_80pct_t_wagenzugmasse"]
    )
    zug["wagenanzahl"] = int(composition["n_coaches"])
    zug["v_max"] = float(composition["v_max_kmh"])

    return body


def query_trassenfinder(start_ds100, end_ds100, composition):
    """Query one route segment for one composition.

    Returns:
        Dictionary with energy consumption, distance and technical travel time.

    Raises:
        TrassenfinderError: on a non-200 response, carrying the API message.
        ValueError: if the API accepts the request but cannot build a route.
    """
    body = build_payload(
        start_ds100,
        end_ds100,
        composition,
        MUTTER_CACHE.get(str(start_ds100), True),
        MUTTER_CACHE.get(str(end_ds100), True),
    )

    response = requests.post(url, json=body, headers=headers)

    if response.status_code != 200:
        raise TrassenfinderError(f"HTTP {response.status_code}: {response.text[:300]}")

    data = response.json()

    # The API returns 200 with a "failure" block when no route can be built.
    if "failure" in data:
        raise ValueError(data["failure"]["message"])

    route = data["result"]["gewichtete_route"]

    return {
        "energy_kwh": route["zusammenfassung"]["energieverbrauch_kwh"],
        "distance_km": route["zusammenfassung"]["weglaenge_hm"] / 10,
        "travel_time_min": route["routenpunkte"][-1]["technische_fahrzeit_info"][
            "ankunft_min"
        ],
    }


print("✓ Query function ready")

## 4. Station pre-flight

Every unique DS100 across both sources is probed once against a fixed reference
station to determine its `mutter` flag, before any collection starts. Two
reasons to do this separately: an invalid code costs one request instead of
eight, and the collection loops then run with a warm cache and no retries.

Stations that resolve neither way are not Betriebsstellen in this
infrastructure version. Their routes are **dropped here**, not carried into the
collection: without a `mutter` flag every one of their requests falls back to
the default and returns HTTP 400, which wastes eight calls per route and buries
the genuine routing failures in the failure table.

The generated route list contains seven such codes: `MURB`, `RBSS`, `BSAL`,
`ADT`, `AOH`, `BWSS`, `AKWS` — Umrathshausen, Baiersbronn Schule, Berlin
Schönhauser Allee, Hamburg Diebsteich, Hamburg-Othmarschen, Berlin Wannsee and
Hamburg Kornweg. Almost all are S-Bahn or local stops that the station export's
name-keyword filter failed to catch, and Trassenfinder does not carry them as
Betriebsstellen in this infrastructure version. They are not correctable
through `DS100_CORRECTIONS`; the fix is to remove them from
`sources/routes_synthetic.csv`.

In [ ]:
REFERENCE_DS100 = "AH"  # Hamburg Hbf, a valid Mutterbetriebsstelle
REFERENCE_MUTTER = True


def resolve_station(ds100, composition):
    """Determine and cache whether a DS100 is a Mutterbetriebsstelle.

    Probes against REFERENCE_DS100, trying mother first. Returns True if the
    station could be resolved, False if Trassenfinder rejects it either way.
    """
    ds100 = str(ds100)

    if ds100 in MUTTER_CACHE:
        return True

    if ds100 == REFERENCE_DS100:
        MUTTER_CACHE[ds100] = REFERENCE_MUTTER
        return True

    for mutter in (True, False):
        body = build_payload(
            REFERENCE_DS100, ds100, composition, REFERENCE_MUTTER, mutter
        )
        response = requests.post(url, json=body, headers=headers)

        # A routing failure still proves the station itself is valid.
        if (
            response.status_code == 200
            or "Ungültige Betriebsstelle" not in response.text
        ):
            MUTTER_CACHE[ds100] = mutter
            return True

    return False


probe_composition = compositions.iloc[0]

stations = sorted(
    {s for df in route_sets.values() for s in df["start_ds100"]}
    | {s for df in route_sets.values() for s in df["end_ds100"]}
)

invalid_stations = [s for s in stations if not resolve_station(s, probe_composition)]

print("Stations resolved:", len(MUTTER_CACHE), "/", len(stations))
print(
    "Mother:",
    sum(MUTTER_CACHE.values()),
    " Child:",
    len(MUTTER_CACHE) - sum(MUTTER_CACHE.values()),
)

if invalid_stations:
    # Drop rather than warn: an unresolved station has no mutter flag, so every
    # one of its requests would fall back to the default and 400. Eight wasted
    # calls per route, and a failure table that hides the real routing failures.
    print("\nINVALID:", invalid_stations)

    for name, df in route_sets.items():
        keep = ~(
            df["start_ds100"].isin(invalid_stations)
            | df["end_ds100"].isin(invalid_stations)
        )
        if (~keep).any():
            print(f"  {name}: dropping {(~keep).sum()} of {len(df)} routes")
            for _, row in df[~keep].iterrows():
                print(
                    f"    - {row['start_ds100']} -> {row['end_ds100']}  "
                    f"({row['start_stop_name']} -> {row['end_stop_name']})"
                )
        route_sets[name] = df[keep].reset_index(drop=True)

    print(
        "\nThese DS100 codes are not Betriebsstellen in this infrastructure "
        "version. Correct them in DS100_CORRECTIONS, or remove them from the "
        "route list under sources/."
    )
    print(
        "Remaining requests:",
        sum(len(df) for df in route_sets.values()) * len(compositions),
    )
else:
    print("\n✓ All stations valid")

## 5. Collect

Each source is queried independently: every route with every composition.
Failures are logged and skipped so one bad route cannot stop the run.

The ONTD set collects clean — every segment is operated, so it is routable by
construction. The generated set is not: roughly 40 of its 100 pairs return
`Es konnte keine Route … gefunden werden`, concentrated on regional stations
that a `D4` locomotive-hauled train cannot reach under these avoidance
settings. Those are logged, not fatal, and the surviving pairs are the ones
worth having.

In [ ]:
def collect(routes, source, compositions):
    """Query every route in one source with every composition.

    Returns:
        (results, failures) as two lists of dictionaries.
    """
    results = []
    failures = []

    print(f"[{source}] {len(routes) * len(compositions)} requests")

    for route_index, (_, route) in enumerate(routes.iterrows(), start=1):
        for _, composition in compositions.iterrows():
            try:
                result = query_trassenfinder(
                    route["start_ds100"], route["end_ds100"], composition
                )

                results.append(
                    {
                        "source": source,
                        "route_name": route["route_name"],
                        "start_stop_name": route["start_stop_name"],
                        "start_ds100": route["start_ds100"],
                        "end_stop_name": route["end_stop_name"],
                        "end_ds100": route["end_ds100"],
                        "composition_id": composition["composition_id"],
                        "n_coaches": composition["n_coaches"],
                        "weight_t": composition[
                            "coaches_gross_weight_80pct_t_wagenzugmasse"
                        ],
                        "length_m": composition["coaches_length_m_wagenzuglaenge"],
                        "v_max_kmh": composition["v_max_kmh"],
                        "energy_kwh": result["energy_kwh"],
                        "distance_km": result["distance_km"],
                        "travel_time_min": result["travel_time_min"],
                    }
                )

            except (TrassenfinderError, ValueError) as e:
                failures.append(
                    {
                        "source": source,
                        "route_name": route["route_name"],
                        "start_ds100": route["start_ds100"],
                        "end_ds100": route["end_ds100"],
                        "composition_id": composition["composition_id"],
                        "error": str(e),
                    }
                )

        if route_index % 10 == 0 or route_index == len(routes):
            print(
                f"  {route_index}/{len(routes)} | ok: {len(results)} | "
                f"failed: {len(failures)}"
            )

    return results, failures


collected = {}
for name, df in route_sets.items():
    collected[name] = collect(df, name, compositions)
    print()

for name, (results, failures) in collected.items():
    print(f"{name:10s} ok: {len(results):4d}  failed: {len(failures):3d}")
    for failure in failures[:3]:
        print(
            f"           {failure['start_ds100']} -> {failure['end_ds100']} | "
            f"{failure['error'][:100]}"
        )

## 6. Save

Each source gets its own pair of files, and both are concatenated into a
combined dataset carrying the `source` column.

`samples_ontd.csv` is the primary training set. Fitting on the generated
pairs or on both means pointing at `samples_synthetic.csv` or
`samples_all.csv`.

In [ ]:
FILENAMES = {
    "ontd": ("samples_ontd.csv", "failures_ontd.csv"),
    "synthetic": ("samples_synthetic.csv", "failures_synthetic.csv"),
}

frames = []
for name, (results, failures) in collected.items():
    results_file, failures_file = FILENAMES[name]

    results_df = pd.DataFrame(results)
    pd.DataFrame(failures).to_csv(DATA_DIR / failures_file, index=False)
    results_df.to_csv(DATA_DIR / results_file, index=False)
    frames.append(results_df)

    print(f"✓ {name}: {len(results_df)} samples -> {results_file}")

combined = pd.concat(frames, ignore_index=True)
combined.to_csv(DATA_DIR / "samples_all.csv", index=False)

print(f"✓ combined: {len(combined)} samples -> samples_all.csv")

## 7. Compare the two samples

The question this run exists to answer: do the two sources describe the same
relationship, or does the ONTD set carry structure specific to how night trains
are routed?

Two things to look at. **Coverage** — the generated set extends the distance
range upward, where ONTD is thin: ONTD reaches 817 km with 80 samples above
400 km, the generated pairs reach 941 km with 216. **Energy intensity at matched
distance** — if kWh/km per band agrees, the samples are interchangeable and can
be pooled. A systematic offset would mean route character, not distance, is
driving part of the fit.

As collected on 2026-08-24 they agree: fleet-weighted kWh/km per composition
matches within 1% across all eight, and a source indicator on the per-km term
is not significant. The samples are poolable.

**Read the generated sample's survivors with care.** Routes fail at very
different rates by station category — 22% for long-distance to long-distance,
58% mixed, 71% regional to regional — because a locomotive-hauled train under
these settings frequently cannot reach branch-line stations at all. `D4` line
class and `knotenbahnhoefe_meiden` are the likely constraints. The surviving
sample is therefore biased towards main-line pairs, which is the part of the
network a night train uses, but it is a selection and not a random draw.

In [ ]:
combined["segment_id"] = (
    combined["route_name"].astype(str)
    + "__"
    + combined["start_ds100"].astype(str)
    + "__"
    + combined["end_ds100"].astype(str)
)
combined["avg_speed_kmh"] = combined["distance_km"] / (combined["travel_time_min"] / 60)
combined["kwh_per_km"] = combined["energy_kwh"] / combined["distance_km"]
combined["band"] = pd.cut(
    combined["distance_km"], [0, 25, 50, 100, 200, 400, 900, 2000]
)

print("Routes and coverage")
print(
    combined.groupby("source")
    .agg(
        samples=("energy_kwh", "size"),
        routes=("segment_id", "nunique"),
        d_min=("distance_km", "min"),
        d_median=("distance_km", "median"),
        d_max=("distance_km", "max"),
        speed_mean=("avg_speed_kmh", "mean"),
    )
    .round(1)
    .to_string()
)

print("\nMedian kWh/km by distance band and source")
print(
    combined.pivot_table(
        index="band",
        columns="source",
        values="kwh_per_km",
        aggfunc="median",
        observed=True,
    )
    .round(2)
    .to_string()
)

print("\nSamples per band and source")
print(
    combined.pivot_table(
        index="band",
        columns="source",
        values="energy_kwh",
        aggfunc="size",
        observed=True,
    ).to_string()
)

print("\nMedian avg speed by band and source")
print(
    combined.pivot_table(
        index="band",
        columns="source",
        values="avg_speed_kmh",
        aggfunc="median",
        observed=True,
    )
    .round(1)
    .to_string()
)

## 8. Quality check

Read back from disk, so this checks the files the regression notebook will
actually load.

In [ ]:
checks = pd.read_csv(DATA_DIR / "samples_all.csv")

checks["segment_id"] = (
    checks["route_name"].astype(str)
    + "__"
    + checks["start_ds100"].astype(str)
    + "__"
    + checks["end_ds100"].astype(str)
)

print("Samples:     ", len(checks))
print("Routes:      ", checks["segment_id"].nunique())
print("Compositions:", checks["composition_id"].nunique())
print("Duplicates:  ", checks.duplicated(["segment_id", "composition_id"]).sum())
print("Missing:     ", checks.isna().sum().sum())

print("\nCompositions per route (want all 8):")
print(
    checks.groupby("segment_id")["composition_id"].nunique().value_counts().to_string()
)

print("\nRanges:")
print(
    checks[["distance_km", "weight_t", "energy_kwh", "travel_time_min"]]
    .describe()
    .round(1)
    .to_string()
)

print("\nFleet-weighted kWh/km by composition and source:")
print(
    checks.groupby(["source", "composition_id"])
    .apply(
        lambda d: d["energy_kwh"].sum() / d["distance_km"].sum(), include_groups=False
    )
    .unstack(0)
    .round(2)
    .to_string()
)

## 9. Handover

### Generated files

| File | Contents |
|---|---|
| `calib/data/samples_ontd.csv` | ONTD night-train segments — the primary training set |
| `calib/data/samples_synthetic.csv` | Generated station pairs |
| `data/processed/samples_all.csv` | Both, with a `source` column |
| `calib/data/failures_*.csv` | Requests that could not be completed, with the API message |

### Columns

`source`, `route_name`, `start_stop_name`, `start_ds100`, `end_stop_name`,
`end_ds100`, `composition_id`, `n_coaches`, `weight_t`, `length_m`,
`v_max_kmh`, `energy_kwh`, `distance_km`, `travel_time_min`

`energy_kwh`, `distance_km` and `travel_time_min` come from Trassenfinder; the
rest are carried through from the input files.

### Known limitations

- **Germany only.** No terrain variation in either source, so no terrain
  coefficient can be estimated. Austrian and Swiss routes are needed for that,
  and neither of these sources provides them.
- **`travel_time_min` is technical running time** — no dwell, no recovery
  margin — and reflects Trassenfinder's own route choice, not the backend's
  routing engine.
- **`weight_t` is a composition-level constant.** Eight compositions, eight
  distinct weights, near-collinear with coach count and train length, so weight
  cannot be separated from other composition attributes.
- **Routes, not country legs.** The backend predicts per country leg; both
  sources are station to station.
- **The generated set is not operational.** Its pairs were sampled for
  diversity, not drawn from timetables, so it should be treated as a robustness
  check on the ONTD fit rather than as evidence about real night-train routing.

### Reproducing

Restart the kernel and run all cells. Collection is idempotent — it overwrites
all output CSVs. Downstream: `02_energy_calibration.ipynb`.